In [86]:
import torch
import pandas as pd

from transformers import AutoTokenizer, T5ForConditionalGeneration

In [87]:
# ============================================================
# Settings
# ============================================================

MODEL_NAME = "/Volumes/BCross/models/t5-small"

NUM_BEAMS = 40
NUM_BEAM_GROUPS = 10
NUM_RETURN_SEQUENCES = 20
DIVERSITY_PENALTY = 0.5

MAX_NEW_TOKENS = 20

# How far we are willing to expand around the original n-gram
MAX_LEFT_EXPANSION = 2
MAX_RIGHT_EXPANSION = 2

In [88]:
# ============================================================
# Device
# ============================================================

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [89]:
# ============================================================
# Load model
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 181.00it/s]


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [90]:
# ============================================================
# Lowercase settings
# ============================================================

LOWERCASE_INPUT = True
LOWERCASE_OUTPUT = True


# ============================================================
# Prepare input
# ============================================================

def prepare_text_and_target(
    text,
    target,
    lowercase_input=False,
):
    """
    Optionally lowercase both the document text and target
    n-gram before tokenisation / generation.

    This ensures that the common n-gram and the text passed
    into T5 use the same casing.
    """

    if lowercase_input:
        text = text.lower()
        target = target.lower()

    return text, target

In [91]:
# ============================================================
# Helpers
# ============================================================

def get_quote_variants(text):
    """
    Generate variants with straight and curly apostrophes.

    This is useful because the model may generate:
        's one
    or:
        ’s one
    """

    variants = {
        text,
        text.replace("'", "’"),
        text.replace("’", "'"),
    }

    return list(variants)


def make_bad_words_ids(original_span):
    """
    Create sequences that T5 is not allowed to generate.

    We include versions with/without a leading space because
    SentencePiece tokenisation can differ depending on whether
    the text occurs at the beginning of a string.
    """

    text_variants = set()

    for variant in get_quote_variants(original_span):

        text_variants.add(variant)

        if not variant.startswith(" "):
            text_variants.add(" " + variant)

    bad_words_ids = []

    for text in text_variants:

        ids = tokenizer(
            text,
            add_special_tokens=False
        ).input_ids

        if len(ids) > 0:
            bad_words_ids.append(ids)

    # Remove duplicate token sequences
    unique = []

    for ids in bad_words_ids:
        if ids not in unique:
            unique.append(ids)

    return unique


def extract_t5_fill(
    token_ids,
    lowercase_output=False,
):
    """
    Extract the generated span between:

        <extra_id_0> ... <extra_id_1>

    Also preserve whether T5's first generated token had a
    SentencePiece whitespace marker.
    """

    extra_id_0 = tokenizer.convert_tokens_to_ids("<extra_id_0>")
    extra_id_1 = tokenizer.convert_tokens_to_ids("<extra_id_1>")

    token_ids = token_ids.tolist()

    try:
        start = token_ids.index(extra_id_0) + 1
    except ValueError:
        start = 1

    try:
        end = token_ids.index(extra_id_1, start)
    except ValueError:
        end = len(token_ids)

    fill_ids = token_ids[start:end]

    if len(fill_ids) == 0:
        return "", False

    # Raw SentencePiece tokens
    fill_tokens = tokenizer.convert_ids_to_tokens(fill_ids)

    # T5 / SentencePiece uses ▁ to indicate a preceding
    # whitespace / word boundary.
    starts_with_space = fill_tokens[0].startswith("▁")

    fill = tokenizer.decode(
        fill_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    if lowercase_output:
        fill = fill.lower()

    return fill, starts_with_space

In [92]:
# ============================================================
# T5 candidate generation
# ============================================================

@torch.no_grad()
def generate_candidates(
    masked_text,
    original_span,
    num_return_sequences=NUM_RETURN_SEQUENCES,
    lowercase_output=False,
):
    """
    Generate alternative fills while preventing T5 from
    reconstructing the original span.
    """

    inputs = tokenizer(
        masked_text,
        return_tensors="pt",
    ).to(device)

    bad_words_ids = make_bad_words_ids(
        original_span
    )

    extra_id_1 = tokenizer.convert_tokens_to_ids(
        "<extra_id_1>"
    )

    outputs = model.generate(
        **inputs,

        do_sample=False,

        num_beams=NUM_BEAMS,
        num_beam_groups=NUM_BEAM_GROUPS,
        diversity_penalty=DIVERSITY_PENALTY,

        num_return_sequences=num_return_sequences,

        bad_words_ids=bad_words_ids,

        eos_token_id=extra_id_1,
        pad_token_id=tokenizer.pad_token_id,

        max_new_tokens=MAX_NEW_TOKENS,

        early_stopping=True,
        renormalize_logits=True,
        
        trust_remote_code=True
    )

    candidates = []

    for output in outputs:

        fill, starts_with_space = extract_t5_fill(
            output,
            lowercase_output=lowercase_output,
        )

        if not fill:
            continue

        candidate_info = {
            "text": fill,
            "starts_with_space": starts_with_space,
        }

        # Deduplicate by candidate text + boundary behaviour
        if candidate_info not in candidates:
            candidates.append(candidate_info)

    return candidates

In [93]:
# ============================================================
# Find target n-gram
# ============================================================

def find_target_span(
    text,
    target,
    occurrence=0,
):
    """
    Find a particular occurrence of the target substring.

    occurrence=0 -> first occurrence
    occurrence=1 -> second occurrence
    etc.
    """

    starts = []

    search_start = 0

    while True:

        index = text.find(
            target,
            search_start
        )

        if index == -1:
            break

        starts.append(index)

        search_start = index + 1

    if len(starts) == 0:
        raise ValueError(
            f"Could not find target {target!r} in text:\n{text}"
        )

    if occurrence >= len(starts):
        raise ValueError(
            f"Requested occurrence {occurrence}, "
            f"but only {len(starts)} occurrence(s) were found."
        )

    start = starts[occurrence]
    end = start + len(target)

    return start, end

In [94]:
# ============================================================
# Convert text to T5 token offsets
# ============================================================

def get_token_offsets(text):
    """
    Return T5 token offsets so that we can expand the target
    span left/right by tokens.
    """

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    offsets = encoded["offset_mapping"]

    return offsets


def get_overlapping_token_indices(
    offsets,
    char_start,
    char_end,
):
    """
    Determine which T5 tokens overlap the target character span.
    """

    indices = []

    for i, (start, end) in enumerate(offsets):

        # Skip zero-length special-token style offsets
        if start == end:
            continue

        overlaps = (
            start < char_end
            and end > char_start
        )

        if overlaps:
            indices.append(i)

    if len(indices) == 0:
        raise ValueError(
            "Could not map target span onto T5 tokens."
        )

    return min(indices), max(indices)

In [95]:
# ============================================================
# Generate expanded spans
# ============================================================

def create_expanded_spans(
    text,
    target,
    occurrence=0,
    max_left=MAX_LEFT_EXPANSION,
    max_right=MAX_RIGHT_EXPANSION,
):
    """
    Generate every span containing the original target while
    expanding up to max_left / max_right T5 tokens.

    Example:

        target = "'s one"

    may produce spans corresponding roughly to:

        's one
        's one of
        It's one
        It's one of
        ...
    """

    char_start, char_end = find_target_span(
        text=text,
        target=target,
        occurrence=occurrence,
    )

    offsets = get_token_offsets(text)

    token_start, token_end = get_overlapping_token_indices(
        offsets=offsets,
        char_start=char_start,
        char_end=char_end,
    )

    spans = []

    for left in range(max_left + 1):

        for right in range(max_right + 1):

            expanded_token_start = max(
                0,
                token_start - left
            )

            expanded_token_end = min(
                len(offsets) - 1,
                token_end + right
            )

            expanded_char_start = offsets[
                expanded_token_start
            ][0]

            expanded_char_end = offsets[
                expanded_token_end
            ][1]

            original_span = text[
                expanded_char_start:
                expanded_char_end
            ]

            masked_text = (
                text[:expanded_char_start]
                + "<extra_id_0>"
                + text[expanded_char_end:]
            )

            span_info = {
                "left_expansion": left,
                "right_expansion": right,
                "char_start": expanded_char_start,
                "char_end": expanded_char_end,
                "original_span": original_span,
                "masked_text": masked_text,
            }

            # Avoid duplicate spans caused by token boundaries
            duplicate = any(
                existing["char_start"] == expanded_char_start
                and existing["char_end"] == expanded_char_end
                for existing in spans
            )

            if not duplicate:
                spans.append(span_info)

    return spans

In [96]:
def reconstruct_with_candidate(
    text,
    char_start,
    char_end,
    candidate,
):
    """
    Replace a character span while repairing whitespace at the
    left and right boundaries.

    Examples
    --------
    Original:
        It's one of the biggest problems.

    Span:
        "'s one"

    Candidate:
        "is one"

    Result:
        "It is one of the biggest problems."

    Candidate:
        "'s one"

    Result:
        "It's one of the biggest problems."
    """

    left = text[:char_start]
    right = text[char_end:]

    candidate = candidate.strip()

    # ========================================================
    # Left boundary
    # ========================================================

    if left and candidate:

        left_char = left[-1]
        candidate_char = candidate[0]

        # If both sides look like ordinary word characters,
        # they need separating whitespace.
        #
        # "It" + "is one"
        # -> "It is one"
        if (
            left_char.isalnum()
            and candidate_char.isalnum()
        ):
            candidate = " " + candidate

        # Apostrophes attach directly:
        #
        # "It" + "'s one"
        # -> "It's one"
        #
        # Includes curly apostrophe.
        elif candidate_char in {"'", "’"}:
            pass

        # If the left side already ends in whitespace,
        # don't add anything.
        elif left_char.isspace():
            pass

    # ========================================================
    # Right boundary
    # ========================================================

    if candidate and right:

        candidate_char = candidate[-1]
        right_char = right[0]

        # If the replacement ends in a word character and the
        # remaining text starts immediately with another word
        # character, add separating whitespace.
        #
        # "among" + "the biggest..."
        # -> "among the biggest..."
        if (
            candidate_char.isalnum()
            and right_char.isalnum()
        ):
            candidate = candidate + " "

        # Existing whitespace means nothing needs changing.
        elif right_char.isspace():
            pass

        # Punctuation should normally remain attached.
        elif right_char in {
            ".",
            ",",
            ";",
            ":",
            "!",
            "?",
            ")",
            "]",
            "}",
            "%",
        }:
            pass

    return (
        left
        + candidate
        + right
    )

In [97]:
def reconstruct_candidate(
    text,
    char_start,
    char_end,
    candidate,
    starts_with_space,
):
    left = text[:char_start]
    right = text[char_end:]

    replacement = candidate

    # SentencePiece says this candidate starts after a
    # whitespace boundary.
    if (
        starts_with_space
        and left
        and not left[-1].isspace()
    ):
        replacement = " " + replacement

    return left + replacement + right

In [98]:
def test_ngram_paraphrases(
    text,
    target,
    occurrence=0,
    max_left=MAX_LEFT_EXPANSION,
    max_right=MAX_RIGHT_EXPANSION,
    lowercase_input=True,
    lowercase_output=True,
):
    """
    Test paraphrase candidates for an n-gram.

    lowercase_input:
        Lowercase the text and target BEFORE tokenisation and
        before passing the masked text into T5.

    lowercase_output:
        Lowercase generated candidate spans and reconstructed
        output text.
    """

    # --------------------------------------------------------
    # Input normalisation
    # --------------------------------------------------------

    working_text, working_target = prepare_text_and_target(
        text=text,
        target=target,
        lowercase_input=lowercase_input,
    )

    spans = create_expanded_spans(
        text=working_text,
        target=working_target,
        occurrence=occurrence,
        max_left=max_left,
        max_right=max_right,
    )

    results = []

    for span_number, span in enumerate(spans):

        print("=" * 80)
        print(
            f"Span {span_number + 1}/{len(spans)}"
        )

        print(
            "Removed:",
            repr(span["original_span"])
        )

        print(
            "Masked:",
            span["masked_text"]
        )

        candidates = generate_candidates(
            masked_text=span["masked_text"],
            original_span=span["original_span"],
            lowercase_output=lowercase_output,
        )

        print("\nCandidates:")

        for rank, candidate_info in enumerate(
            candidates,
            start=1,
        ):

            candidate = candidate_info["text"]
            starts_with_space = candidate_info["starts_with_space"]

            reconstructed = reconstruct_candidate(
                text=working_text,
                char_start=span["char_start"],
                char_end=span["char_end"],
                candidate=candidate,
                starts_with_space=starts_with_space,
            )

            if lowercase_output:
                reconstructed = reconstructed.lower()

            print(
                f"{rank:>2}. {candidate!r}"
            )

            print(
                "    ",
                reconstructed
            )

            results.append({
                "span_number": span_number + 1,
                "left_expansion": span["left_expansion"],
                "right_expansion": span["right_expansion"],
                "original_span": span["original_span"],
                "masked_text": span["masked_text"],
                "candidate_rank": rank,
                "candidate": candidate,
                "reconstructed_text": reconstructed,
                "lowercase_input": lowercase_input,
                "lowercase_output": lowercase_output,
                "starts_with_space": starts_with_space,
            })

        print()

    return pd.DataFrame(results)

In [99]:
# ============================================================
# Example
# ============================================================

text = "It's one of the biggest problems we face."

target = "'s one"

results_df = test_ngram_paraphrases(
    text=text,
    target=target,
    occurrence=0,

    # Start small for testing
    max_left=2,
    max_right=2,
    
    # lowercase
    lowercase_input=True,
    lowercase_output=True
)

Span 1/6
Removed: "'s one"
Masked: it<extra_id_0> of the biggest problems we face.


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Candidates:
 1. 'is one'
     it is one of the biggest problems we face.
 2. 'one'
     it one of the biggest problems we face.
 3. 'as one'
     it as one of the biggest problems we face.
 4. ', one'
     it, one of the biggest problems we face.
 5. '’s one'
     it’s one of the biggest problems we face.
 6. 'a few'
     it a few of the biggest problems we face.
 7. '. it is one'
     it. it is one of the biggest problems we face.
 8. ', is one'
     it, is one of the biggest problems we face.
 9. '. this is one'
     it. this is one of the biggest problems we face.
10. '. one'
     it. one of the biggest problems we face.

Span 2/6
Removed: "'s one of"
Masked: it<extra_id_0> the biggest problems we face.


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Candidates:
 1. 'is one of'
     it is one of the biggest problems we face.
 2. 'one of'
     it one of the biggest problems we face.
 3. 'are one of'
     it are one of the biggest problems we face.
 4. ', one of'
     it, one of the biggest problems we face.
 5. 'is among'
     it is among the biggest problems we face.
 6. 'is'
     it is the biggest problems we face.
 7. '. it is one of'
     it. it is one of the biggest problems we face.
 8. '. we are one of'
     it. we are one of the biggest problems we face.
 9. '. we have one of'
     it. we have one of the biggest problems we face.
10. 'is one'
     it is one the biggest problems we face.
11. '–'
     it – the biggest problems we face.
12. '’s'
     it’s the biggest problems we face.
13. 'to be'
     it to be the biggest problems we face.

Span 3/6
Removed: "'s one of the"
Masked: it<extra_id_0> biggest problems we face.


...


Candidates:
 1. 'one'
     one of the biggest problems we face.
 2. 'are one'
     are one of the biggest problems we face.
 3. 'is one'
     is one of the biggest problems we face.
 4. 'are some'
     are some of the biggest problems we face.
 5. 'some'
     some of the biggest problems we face.

Span 5/6
Removed: "it's one of"
Masked: <extra_id_0> the biggest problems we face.


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Candidates:
 1. 'are'
     are the biggest problems we face.
 2. 'one of'
     one of the biggest problems we face.
 3. 'are some of'
     are some of the biggest problems we face.
 4. 'are one of'
     are one of the biggest problems we face.
 5. 'are probably'
     are probably the biggest problems we face.
 6. 'face'
     face the biggest problems we face.
 7. 'of'
     of the biggest problems we face.
 8. 'among'
     among the biggest problems we face.
 9. ', are'
     , are the biggest problems we face.
10. '–'
     – the biggest problems we face.
11. 'some of'
     some of the biggest problems we face.

Span 6/6
Removed: "it's one of the"
Masked: <extra_id_0> biggest problems we face.


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Candidates:
 1. 'the'
     the biggest problems we face.
 2. 'of the'
     of the biggest problems we face.
 3. 'are the'
     are the biggest problems we face.
 4. 'of our'
     of our biggest problems we face.
 5. 'have one of the'
     have one of the biggest problems we face.
 6. 'have some of the'
     have some of the biggest problems we face.
 7. 'one of the'
     one of the biggest problems we face.
 8. ', we have one of the'
     , we have one of the biggest problems we face.
 9. ', we face one of the'
     , we face one of the biggest problems we face.
10. ', we face some of the'
     , we face some of the biggest problems we face.



In [100]:
# ============================================================
# View final dataframe
# ============================================================

print("\n")
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)

print(
    results_df[
        [
            "left_expansion",
            "right_expansion",
            "original_span",
            "candidate_rank",
            "candidate",
            "reconstructed_text",
        ]
    ].to_string(index=False)
)



FINAL RESULTS
 left_expansion  right_expansion   original_span  candidate_rank             candidate                               reconstructed_text
              0                0          's one               1                is one       it is one of the biggest problems we face.
              0                0          's one               2                   one          it one of the biggest problems we face.
              0                0          's one               3                as one       it as one of the biggest problems we face.
              0                0          's one               4                 , one         it, one of the biggest problems we face.
              0                0          's one               5                ’s one        it’s one of the biggest problems we face.
              0                0          's one               6                 a few        it a few of the biggest problems we face.
              0                0